# Google Maps Timeline — Location Heatmap

Maps every location recorded in a Google Maps **Timeline** export: a world overview,
a zoom on the area where most points fall, a city-level zoom and an interactive heatmap.

**Where the data comes from.** Timeline is now stored on the phone, not in Takeout.
Export it from the phone (Android: *Settings → Location → Location services → Timeline →
Export Timeline data*) and put the JSON file in a `Timeline/` folder next to this notebook.
The file name depends on the phone's language (`Timeline.json`, `Cronología.json`, …);
the notebook finds it by its contents, not its name. An older Takeout
`Records.json` (Location History) also works.

**Privacy.** The export and every output of this notebook are personal. `*.json` is
gitignored and notebook outputs are stripped on commit. The basemap tiles are downloaded
from Esri, so its servers see which
map area is being drawn.

## Setup

In [ ]:
import html
import json
import re
import unicodedata
from pathlib import Path

import contextily as ctx
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML
from matplotlib.colors import LinearSegmentedColormap

pd.set_option('display.width', 160)

def _norm(text):
    """Lowercase and strip accents so Spanish/English names both match."""
    text = unicodedata.normalize('NFKD', str(text))
    return ''.join(c for c in text if not unicodedata.combining(c)).strip().lower()

# Folder names the export may live in, after _norm().
TIMELINE_DIRS = ('timeline', 'cronologia', 'location history', 'historial de ubicaciones')

def find_timeline_files(start=None):
    """Every *.json inside a Timeline-like folder, searching up from the notebook."""
    start = Path(start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        hits = []
        for d in [base, base / 'Takeout']:
            if not d.is_dir():
                continue
            for sub in d.iterdir():
                if sub.is_dir() and _norm(sub.name) in TIMELINE_DIRS:
                    hits += sorted(sub.rglob('*.json'))
        if hits:
            return hits
    raise FileNotFoundError('no Timeline/ folder with JSON files above ' + str(start))

TIMELINE_FILES = find_timeline_files()
for p in TIMELINE_FILES:
    print(f'{p.parent.name}/{p.name}  ({p.stat().st_size / 1e6:,.1f} MB)')

### Plot style

Same palette as the Google Fit notebook: fixed categorical order and a single-hue blue
ramp for density.

In [ ]:
SURFACE   = '#fcfcfb'
INK       = '#0b0b0b'
INK_SOFT  = '#52514e'
INK_MUTED = '#898781'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
BLUES = LinearSegmentedColormap.from_list(
    'fit_blue',
    ['#eef5fd', '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'],
)

mpl.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'figure.dpi': 110,
    'font.family': 'sans-serif', 'font.sans-serif': ['Segoe UI', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9, 'axes.titlecolor': INK, 'axes.titlesize': 11,
    'axes.titleweight': 'semibold', 'axes.titlelocation': 'left',
    'legend.frameon': False, 'legend.fontsize': 8.5, 'legend.labelcolor': INK_SOFT,
})

# Light grey, no labels, no API key needed (CARTO tiles now require one).
BASEMAP = ctx.providers.Esri.WorldGrayCanvas

def map_chrome(ax, title, subtitle=None):
    """Title/subtitle like the Fit notebook; no axes, ticks or frame on a map."""
    ax.set_title(title, pad=22 if subtitle else 10)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1), xycoords='axes fraction', xytext=(0, 7),
                    textcoords='offset points', fontsize=8.5, color=INK_MUTED, va='bottom')
    ax.set_axis_off()
    return ax

<a id="1"></a>
## 1. Load every point

Three export formats are understood:

| Format | Where it comes from | Shape |
|---|---|---|
| on-device, Android | Timeline export on the phone | `{"semanticSegments": [...], "rawSignals": [...]}` |
| on-device, iOS | Timeline export in the Maps app | a top-level list of segments, coordinates as `geo:lat,lng` |
| legacy | Takeout *Location History* `Records.json` | `{"locations": [{"latitudeE7": ...}]}` |

Each location becomes one row with a `source`:

- `path`: points along a travelled route (`timelinePath`)
- `visit`: the place of a stay (home, work, a shop…)
- `trip`: the start and end of a movement segment
- `raw`: raw GPS fixes (`rawSignals` / `Records.json`), with their accuracy

In [ ]:
_NUM = re.compile(r'-?\d+(?:\.\d+)?')

def parse_latlng(value):
    """'40.4°, -3.7°' / 'geo:40.4,-3.7' / {'latLng': ...} -> (lat, lon), or None."""
    if isinstance(value, dict):
        value = value.get('latLng') or value.get('LatLng') or value.get('placeLocation')
    if not isinstance(value, str):
        return None
    nums = _NUM.findall(value)
    if len(nums) < 2:
        return None
    lat, lon = float(nums[0]), float(nums[1])
    return (lat, lon) if -90 <= lat <= 90 and -180 <= lon <= 180 else None

def _add(rows, source, value, when, accuracy=np.nan):
    ll = parse_latlng(value)
    if ll:
        rows.append((when, ll[0], ll[1], source, accuracy))

def _segments(rows, segments):
    for s in segments:
        start = s.get('startTime')
        for p in s.get('timelinePath') or []:
            _add(rows, 'path', p.get('point'), p.get('time', start))
        visit = (s.get('visit') or {}).get('topCandidate') or {}
        _add(rows, 'visit', visit.get('placeLocation'), start)
        act = s.get('activity') or {}
        _add(rows, 'trip', act.get('start'), start)
        _add(rows, 'trip', act.get('end'), s.get('endTime', start))

def load_timeline(path):
    data = json.loads(path.read_text(encoding='utf-8'))
    rows = []
    if isinstance(data, list):                                  # iOS
        _segments(rows, data)
    elif 'semanticSegments' in data or 'rawSignals' in data:    # Android
        _segments(rows, data.get('semanticSegments', []))
        for r in data.get('rawSignals', []):
            pos = r.get('position')
            if pos:
                _add(rows, 'raw', pos.get('LatLng'), pos.get('timestamp'),
                     pos.get('accuracyMeters', np.nan))
    elif 'locations' in data:                                   # legacy Records.json
        for r in data['locations']:
            if 'latitudeE7' in r and 'longitudeE7' in r:
                rows.append((r.get('timestamp'), r['latitudeE7'] / 1e7, r['longitudeE7'] / 1e7,
                             'raw', r.get('accuracy', np.nan)))
    else:
        print(f'  skipped {path.name}: not a Timeline export')
    return rows

rows = []
for p in TIMELINE_FILES:
    rows += load_timeline(p)
pts = pd.DataFrame(rows, columns=['time', 'lat', 'lon', 'source', 'accuracy_m'])
pts['time'] = pd.to_datetime(pts['time'], utc=True, format='ISO8601', errors='coerce')
pts['year'] = pts['time'].dt.year

# Raw fixes with a very large error circle add blur, not information.
pts = pts[~(pts.accuracy_m > 500)].drop_duplicates(['time', 'lat', 'lon']).reset_index(drop=True)

print(f'{len(pts):,} points, {pts.time.min():%Y-%m-%d} -> {pts.time.max():%Y-%m-%d}')
pts.groupby('source').agg(points=('lat', 'size'), first=('time', 'min'), last=('time', 'max'))

In [ ]:
per_year = pts.pivot_table(index='year', columns='source', values='lat', aggfunc='size').fillna(0)
fig, ax = plt.subplots(figsize=(11, 3.6))
order = [s for s in ['path', 'visit', 'trip', 'raw'] if s in per_year]
bottom = np.zeros(len(per_year))
for i, src in enumerate(order):
    ax.bar(per_year.index, per_year[src], bottom=bottom, color=SERIES[i], width=0.7,
           label=src, edgecolor=SURFACE, lw=1)
    bottom += per_year[src].values
ax.set_xticks(per_year.index)
ax.legend(loc='upper left', ncols=len(order))
ax.grid(axis='y', color='#e1e0d9')
ax.set_axisbelow(True)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.set_title('Recorded points per year', pad=22)
ax.annotate('stacked by kind of record', xy=(0, 1), xycoords='axes fraction', xytext=(0, 7),
            textcoords='offset points', fontsize=8.5, color=INK_MUTED, va='bottom')
plt.show()

<a id="2"></a>
## 2. World map

Every point at low opacity on a world basemap: a single point is faint, and
places visited again and again build up into solid colour.

In [ ]:
R = 6378137.0   # Web Mercator (EPSG:3857), the projection the tiles are drawn in

def to_mercator(lat, lon):
    lat = np.clip(lat, -85.0511, 85.0511)
    x = np.radians(lon) * R
    y = np.log(np.tan(np.pi / 4 + np.radians(lat) / 2)) * R
    return x, y

pts['x'], pts['y'] = to_mercator(pts.lat.values, pts.lon.values)

def set_extent(ax, lat_min, lat_max, lon_min, lon_max):
    (x0, x1), (y0, y1) = [np.array(v) for v in zip(*[to_mercator(lat_min, lon_min),
                                                     to_mercator(lat_max, lon_max)])]
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.2))
ax.scatter(pts.x, pts.y, s=6, color=SERIES[0], alpha=0.08, linewidths=0, zorder=3)
set_extent(ax, -58, 78, -180, 180)   # after plotting, so autoscaling can't undo it
ctx.add_basemap(ax, source=BASEMAP, zoom=2, attribution_size=6)
map_chrome(ax, 'Everywhere the phone has been',
           f'{len(pts):,} points · darker = more points on top of each other')
plt.show()

<a id="3"></a>
## 3. Main region

The world view squashes daily life into a few dots. Zooming to the box that holds the
central 96% of points shows the area most of the data covers. The left panel keeps the
low-opacity points; the right one bins them into hexagons coloured by count (log scale),
so busy places stand out without saturating.

In [ ]:
def core_extent(df, q=0.02, pad=0.08):
    lat0, lat1 = df.lat.quantile([q, 1 - q])
    lon0, lon1 = df.lon.quantile([q, 1 - q])
    dlat, dlon = (lat1 - lat0) * pad + 0.01, (lon1 - lon0) * pad + 0.01
    return lat0 - dlat, lat1 + dlat, lon0 - dlon, lon1 + dlon

def region_maps(df, ext, title, point_alpha=0.05, gridsize=90):
    """Points and hexagon density side by side for the box ext = (lat0, lat1, lon0, lon1)."""
    lat0, lat1, lon0, lon1 = ext
    sub = df[df.lat.between(lat0, lat1) & df.lon.between(lon0, lon1)]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.8), layout='constrained')
    ax1.scatter(sub.x, sub.y, s=4, color=SERIES[0], alpha=point_alpha, linewidths=0, zorder=3)
    hb = ax2.hexbin(sub.x, sub.y, gridsize=gridsize, bins='log', mincnt=1, cmap=BLUES,
                    alpha=0.85, linewidths=0, zorder=3)
    for ax in (ax1, ax2):
        set_extent(ax, *ext)
        ax.set_aspect('equal')
        ctx.add_basemap(ax, source=BASEMAP, attribution_size=6)
    map_chrome(ax1, f'{title}: points',
               f'{len(sub):,} of {len(df):,} points · low opacity, overlaps darken')
    map_chrome(ax2, f'{title}: density', 'points per hexagon · log scale')
    # One colorbar for both axes keeps the two panels the same size.
    cb = fig.colorbar(hb, ax=[ax1, ax2], fraction=0.025, pad=0.01, shrink=0.7)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=7.5)
    plt.show()
    return sub

inside = region_maps(pts, core_extent(pts), 'Main region')

<a id="4"></a>
## 4. City

The single busiest spot (the 0.05° cell with the most points), with a box of
roughly 30 × 30 km around it: the street-level picture of daily routes.

In [ ]:
cell = (pts.assign(clat=(pts.lat / 0.05).round(), clon=(pts.lon / 0.05).round())
        .groupby(['clat', 'clon']).size().idxmax())
c_lat, c_lon = cell[0] * 0.05, cell[1] * 0.05
half_lat = 0.14
half_lon = half_lat / np.cos(np.radians(c_lat))   # same ground distance east-west
city = region_maps(pts, (c_lat - half_lat, c_lat + half_lat, c_lon - half_lon, c_lon + half_lon),
                   'City', point_alpha=0.04, gridsize=70)

<a id="5"></a>
## 5. Interactive heatmap

A full-page map built from [templates/timeline_heatmap.html](templates/timeline_heatmap.html),
with the points written into it. Its layout and controls follow
[ry-li/google-maps-timeline-heatmap](https://github.com/ry-li/google-maps-timeline-heatmap).
It is shown below and saved to `out/timeline_heatmap.html` (gitignored). Open that file in a
browser for the full-screen version.

The panel in the top-right corner changes the map live:

- **Radius / Blur**: size and softness of each heat blob.
- **Intensity**: the zoom level at which heat reaches full strength. Below it, heat is
  halved for each zoom level, so a lower value makes the map hotter.
- **Compression**: a root applied to the summed heat (1 = raw, 2 = square root…), so
  rarely visited places stay visible next to home.
- **Base map**: dark, light grey or a colourful street map, all from Esri, with no API key.

`HEATMAP_SETTINGS` sets the values the page starts with.

In [ ]:
# Starting values for the page's sliders; all of them can be changed live in the browser.
HEATMAP_SETTINGS = {
    'radius': 15,           # px; size of each blob
    'blur': 15,             # px; softness of the blob edges
    'intensity': 16,        # zoom level where heat is at full strength (lower = hotter)
    'compression': 2,       # root applied to the heat: 1 = raw, 2 = square root, higher = flatter
    'pointWeight': 0.05,    # heat added by one point; 20 points on one spot reach the top colour
    'minOpacity': 0.05,
    'base': 'dark',         # 'dark', 'light' or 'streets'
    'gradient': {0.0: 'blue', 0.4: 'cyan', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red'},
}
# Points are merged into ~11 m cells (with their count) only to keep the HTML small;
# that is far below the size of a heat blob at any zoom.
CELL_DEG = 0.0001

cells = (pts.assign(clat=(pts.lat / CELL_DEG).round() * CELL_DEG,
                    clon=(pts.lon / CELL_DEG).round() * CELL_DEG)
         .groupby(['clat', 'clon']).size().rename('n').reset_index())
print(f'{len(pts):,} points -> {len(cells):,} cells of ~11 m')

payload = {
    'cells': [[round(a, 5), round(b, 5), int(n)] for a, b, n in cells.itertuples(index=False)],
    'points': len(pts),
    'center': [c_lat, c_lon],   # the busiest spot found in section 4
    'zoom': 12,
    'settings': HEATMAP_SETTINGS,
}
template = Path('templates/timeline_heatmap.html').read_text(encoding='utf-8')
page = template.replace('/*__DATA__*/null', json.dumps(payload, separators=(',', ':')))

out = Path('out')
out.mkdir(exist_ok=True)
(out / 'timeline_heatmap.html').write_text(page, encoding='utf-8')
print(f'saved out/timeline_heatmap.html ({len(page) / 1e6:.1f} MB)')

# Show the same page inline. srcdoc keeps it self-contained, without a server.
HTML(f'<div><iframe srcdoc="{html.escape(page)}" style="width:100%;height:640px;border:0"></iframe></div>')

## Notes

- Every point counts once, whatever kind it is. A long stay at home is one `visit` point,
  while a 20-minute walk can leave dozens of `path` points, so the maps show where the
  phone *moved and was recorded*, not where time was spent.
- Weighting visits by their duration, or colouring by year or by trip mode
  (walking / cycling / driving, available in `activity.topCandidate`), are natural next
  steps.